# Pandas DataFrames

> 📘 **Python Mastery** · Module 11 — Pandas · Lesson 2/7

A DataFrame is a full table — rows *and* columns of labeled data. It is the object you will spend most of your data-science life holding, so this lesson builds it up carefully and ends with the selection tool you must truly master: `loc` vs `iloc`.

## 🎯 Learning Objectives

- **Create** DataFrames from a dict-of-lists and from a list-of-dicts, and predict how missing keys behave.
- **Inspect** any table quickly with `head()`, `tail()`, `sample()`, `shape`, `columns`, `dtypes` and `info()`.
- **Distinguish** selecting one column (a Series) from selecting a list of columns (a smaller DataFrame).
- **Add**, derive, rename and delete columns and rows (`axis=` demystified).
- **Apply** `.loc` vs `.iloc` correctly for label-based vs position-based selection, slicing and assignment.
- **Promote** an ID column to a meaningful index with `set_index()`.

## 1. What Is a DataFrame?

If a Series is one labeled column, a **DataFrame** is a dict of Series sharing one index — in other words, a spreadsheet or SQL table living in Python: named columns down the page, labeled rows across it.

> 🔍 **Under the Hood:** A DataFrame stores each column as its own NumPy array inside a structure pandas calls the *block manager*. Columns of the same dtype are grouped into blocks, which is why `df.dtypes` matters: one `int64` column and one `str` column live in separate memory blocks, not one mixed array. That is also why a DataFrame can hold mixed types while each individual column cannot.

**Syntax:**

```python
pd.DataFrame(data, index=None, columns=None)
# data: dict of lists, list of dicts, list of lists, another DataFrame...
```

## 2. Creating a DataFrame

The two shapes you will use daily:

1. **Dict of lists** — keys become *column names*, each list is one column.
2. **List of dicts** — each dict is one *row*; keys become column names. If a row lacks a key, pandas fills that cell with `NaN`.

Both need equal-length data per column; mismatched lengths raise `ValueError`. The Bangladesh-students table below will be our running example.

In [ ]:
import pandas as pd

# Style 1: dict of lists -> keys are COLUMN names
students = pd.DataFrame({
    "student_id": ["S001", "S002", "S003", "S004"],
    "name":       ["Sarah", "Rafi", "Nabila", "Karim"],
    "age":        [22, 25, 21, 23],
    "city":       ["Dhaka", "Sylhet", "Dhaka", "Chattogram"],
    "score":      [88, 92, 79, 95],
})
print(students)
print()
print(type(students))

In [ ]:
import pandas as pd

# Style 2: list of dicts -> each dict is ONE ROW
# Nabila has no 'age' key -> that cell becomes NaN automatically
enrolments = pd.DataFrame([
    {"student_id": "S005", "name": "Mim",     "city": "Khulna"},
    {"student_id": "S006", "name": "Tanvir",  "city": "Rajshahi", "age": 24},
])
print(enrolments)

orders = pd.DataFrame({"product": ["pen", "book"], "qty": [3, 2]})   # equal lengths required
# pd.DataFrame({"product": ["pen", "book"], "qty": [3]})   <- ValueError: length mismatch

## 3. First Look at Any Table

Before analyzing anything, *look* at it. These six calls answer "what do I actually have?" in ten seconds:

| Call | Tells you |
|---|---|
| `df.head(n)` / `df.tail(n)` | first / last n rows |
| `df.sample(n, random_state=...)` | random peek (seed it to stay reproducible) |
| `df.shape` | `(rows, cols)` |
| `df.columns` / `df.index` | column names / row labels |
| `df.dtypes` | type of every column |
| `df.info()` | all of the above + memory + non-null counts |

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "student_id": ["S001", "S002", "S003", "S004", "S005"],
    "name":       ["Sarah", "Rafi", "Nabila", "Karim", "Mim"],
    "age":        [22, 25, 21, 23, 20],
    "city":       ["Dhaka", "Sylhet", "Dhaka", "Chattogram", "Khulna"],
    "score":      [88, 92, 79, 95, 84],
})

print(students.head(2))      # top rows (default 5)
print()
print(students.tail(1))      # bottom row
print()
print(students.sample(2, random_state=42))   # deterministic random peek
print()
print("shape:", students.shape, "| columns:", list(students.columns))
print(students.dtypes)

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "student_id": ["S001", "S002", "S003", "S004"],
    "name":       ["Sarah", "Rafi", "Nabila", "Karim"],
    "age":        [22, 25, None, 23],          # one missing age on purpose
    "city":       ["Dhaka", "Sylhet", "Dhaka", "Chattogram"],
    "score":      [88, 92, 79, 95],
})

students.info()   # prints a report: dtypes, non-null counts, memory use
# RangeIndex tells us rows are still labeled 0..n-1

## 4. Selecting Columns: Series vs DataFrame

Square brackets on a DataFrame select **columns** — and what you get back depends on what you ask for:

- `df["col"]` → a single **Series**
- `df[["col1", "col2"]]` → note the *inner* list! → a new **DataFrame**

Forgetting the inner brackets is a classic slip: `df[["name"]]` and `df["name"]` look similar but print differently (`DataFrame` has a tabular header, a `Series` shows its dtype).

**Syntax:**

```python
s  = df["col"]              # Series  (one column)
df = df[["a", "b"]]         # DataFrame (subset of columns, order = yours)
```

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "student_id": ["S001", "S002", "S003"],
    "name":       ["Sarah", "Rafi", "Nabila"],
    "city":       ["Dhaka", "Sylhet", "Dhaka"],
    "score":      [88, 92, 79],
})

one = students["score"]          # ONE column -> Series
many = students[["name", "score"]]  # LIST of columns -> DataFrame

print(one)                       # note the Name: score footer
print()
print(type(one), type(many))
print()
print(many)
print()
print(students[["score", "name"]])   # your order wins -- score first now

## 5. Creating and Deleting Columns

New columns usually come from old ones. Assign to a brand-new name and pandas adds the column at the right edge; every existing row gets the computed value. Delete with `drop()` — and mind the **`axis=`**: columns live on `axis=1` ("across" the table's width), rows on `axis=0`.

`drop()` does *not* modify the original unless you reassign (or pass `inplace=True`, which we avoid — reassigning keeps data flow visible).

**Syntax:**

```python
df["new"] = df["a"] * df["b"]            # derived column
df.drop("col", axis=1)                   # drop a column (returns a copy!)
df.drop(["c0", "c1"], axis=1)            # several at once
df.drop(0, axis=0)                       # drop ROW labeled 0 (axis=0 is default)
```

In [ ]:
import pandas as pd

cart = pd.DataFrame({
    "item":  ["pen", "notebook", "backpack"],
    "price": [25, 120, 1450],           # taka
    "qty":   [4, 2, 1],
})

cart["total"] = cart["price"] * cart["qty"]        # derived column
cart["in_budget"] = cart["total"] <= 500           # derived BOOLEAN column
print(cart)

kept = cart.drop("in_budget", axis=1)              # axis=1 -> column
kept = kept.drop(["price", "qty"], axis=1)         # several columns
print()
print(kept)
print()
print(cart.columns.tolist())   # original untouched by drop -- we reassigned

In [ ]:
import pandas as pd

cart = pd.DataFrame({
    "item":  ["pen", "notebook", "backpack"],
    "price": [25, 120, 1450],
    "qty":   [4, 2, 1],
})

by_row = cart.drop(0)                 # default axis=0 -> drops ROW label 0
print(by_row)
print()
print(cart.drop(index=[0, 2]))        # explicit form: same idea, reads clearly

## 6. Renaming Columns

Two spellings of the same idea — pick one style and stay consistent:

- `df.rename(columns={...})` — the popular, self-documenting form.
- `df.rename({...}, axis="columns")` — same mapping, but `axis=` names the target; use `axis="index"` (or `axis=0`) when renaming **row labels** instead.

Like `drop`, `rename` returns a **copy**: assign it back.

**Syntax:**

```python
df.rename(columns={"old": "new"})            # rename columns by mapping
df.rename({"old": "new"}, axis="columns")    # identical result, axis= spelling
df.rename(columns=str.upper)                 # apply a function to EVERY column name
```

In [ ]:
import pandas as pd

weather = pd.DataFrame({"city": ["Dhaka", "Sylhet"], "temp_c": [33, 29], "rain_mm": [12, 48]})
print(weather.columns.tolist())

renamed = weather.rename(columns={"temp_c": "temperature_c"})
same_thing = weather.rename({"temp_c": "temperature_c"}, axis="columns")   # axis= spelling

upper = weather.rename(columns=str.upper)               # function over every name
print(list(renamed.columns), "|", list(same_thing.columns), "|", list(upper.columns))

row_named = weather.rename({0: "monday"}, axis="index")  # renaming ROW labels works too
print(row_named.index.tolist())

## 7. ⚠️ `loc` vs `iloc` — The Masterclass

Row selection comes in two flavors and mixing them up produces either errors or silently wrong answers:

| | `.loc` | `.iloc` |
|---|---|---|
| Selects by | **label** | integer **position** |
| Slice end | **INCLUSIVE** | exclusive (like Python lists) |
| Accepts | labels, boolean masks, functions | integers, slices, lists of ints |

⚠️ **The slice surprise:** `df.loc[0:2]` returns rows labeled 0, 1 **and 2** (three rows!), while `df.iloc[0:2]` returns positions 0 and 1 (two rows). Label slices include their endpoint.

**Syntax:**

```python
df.loc[row_label]                     # one row (as a Series)
df.loc[row_start:row_end]             # label slice, END INCLUDED
df.loc[row_sel, col_sel]              # rows AND columns
df.loc[df["score"] >= 80]             # boolean mask through loc
df.iloc[pos, pos]                     # everything by position, end EXCLUDED
```

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "city":  ["Dhaka", "Sylhet", "Khulna", "Barishal"],
    "temp":  [33, 29, 31, 30],
}, index=["Sat", "Sun", "Mon", "Tue"])          # string labels make the difference vivid

print(df.loc["Sat":"Mon"])      # LABEL slice: Sat, Sun, Mon  -> 3 rows, END INCLUDED
print()
print(df.iloc[0:2])             # POSITION slice: rows 0,1     -> 2 rows, end excluded

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "student_id": ["S001", "S002", "S003"],
    "name":       ["Sarah", "Rafi", "Nabila"],
    "score":      [88, 92, 79],
})

row = students.loc[1]                  # row LABELED 1 (a Series: column names become labels)
print(row)
print()
print(students.loc[1, "name"])         # scalar: row 1, column 'name'
print(students.loc[[0, 2], ["name", "score"]])   # list of labels both ways
print()
print(students.iloc[-1])               # last row BY POSITION (.loc[-1] would fail: no such label!)

### 7.1 Boolean Masking Through `.loc`

The professional pattern: `df.loc[mask]` filters rows, and `df.loc[mask, cols]` filters rows **and** picks columns in one shot. It replaces clunky chains and is the only safe way to *write* values conditionally (next section).

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "name":  ["Sarah", "Rafi", "Nabila", "Karim"],
    "city":  ["Dhaka", "Sylhet", "Dhaka", "Chattogram"],
    "score": [88, 92, 79, 95],
})

mask = students["score"] >= 85
print(students.loc[mask])                        # rows meeting the condition
print()
print(students.loc[mask, ["name", "score"]])     # ...but only these two columns
print()
print(students.loc[students["city"] == "Dhaka", "name"])   # one column of matching rows

### 7.2 Setting Values with `.loc` — and Why Chained Assignment Is Dead

To change data, address it exactly like you read it: `df.loc[mask, col] = value`. In modern pandas (**Copy-on-Write**, the default since 3.0) the old chained shortcut **silently stops working** — `df[mask]["score"] = 99` modifies a temporary copy and your real table never changes. One more reason `.loc` is not optional.

**Syntax:**

```python
df.loc[mask, "col"] = value            # write where the mask is True
df.loc[row_label, "col"] = value       # write one cell
```

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "name":  ["Sarah", "Rafi", "Nabila", "Karim"],
    "grade": ["B", "A", "C", "A"],
    "score": [88, 92, 79, 95],
})

# give everyone scoring 90+ an A+
students.loc[students["score"] >= 90, "grade"] = "A+"
print(students)

students.loc[2, "score"] = 81          # fix one cell precisely

broken = students.copy()
broken[broken["score"] > 90]["grade"] = "X"   # chained assignment: NO effect under CoW
print()
print(broken["grade"].tolist(), "<- 'Karim' unchanged: the chained write hit a throwaway copy")

## 8. Meaningful Rows: `set_index()`

A RangeIndex of `0,1,2...` wastes your best labeling opportunity. When a column holds unique IDs, promote it: `set_index("col")` makes those values the row labels, so `.loc["S002"]` becomes a readable query and joins later lessons get easier.

Its mirror is `reset_index()` (labels back to 0..n, the old index returned as a normal column).

**Example:**

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "student_id": ["S001", "S002", "S003"],
    "name":       ["Sarah", "Rafi", "Nabila"],
    "score":      [88, 92, 79],
})

indexed = students.set_index("student_id")
print(indexed)
print()
print(indexed.loc["S002"])            # fetch a student by ID -- no position counting
print()
print(indexed.reset_index())          # undo: ID returns to being a regular column

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| `df[["name"]]` vs `df["name"]` confusion | One gives a 1-column DataFrame, the other a Series — downstream methods differ | Decide deliberately; double brackets when you need a table |
| `df.loc[0:2]` assumed exclusive | Label slices are INCLUSIVE — you quietly get an extra row | Remember: `.loc` includes the endpoint, `.iloc` does not |
| Chained assignment `df[m]["col"] = x` | Under Copy-on-Write this writes to a copy; your table is untouched (no error!) | Always `df.loc[m, "col"] = x` |
| `drop("colname")` without `axis=1` | Default is rows; pandas looks for a row label and raises `KeyError` | Pass `axis="columns"` or `drop(columns=[...])` |
| `df.score` attribute access | Breaks for names with spaces, or columns that shadow methods (`count`) | Stick to `df["score"]` everywhere |

## 💡 Best Practices & Pro Tips

- **Read before you compute:** make `df.shape`, `df.head()` and `df.info()` a reflex after creating or loading any table.
- **Write only through `.loc`.** It is precise, CoW-proof, and self-documenting about *which* rows changed.
- **Set a meaningful index early** (IDs, timestamps). Everything from merging to time series gets cleaner.
- Reassign results instead of mutating in place — `df = df.drop(...)` makes the data flow readable and avoids `inplace=True` pitfalls.
- 🤖 **AI-engineering relevance:** datasets arrive as DataFrames and get sliced with `loc`/`iloc` into train/test splits; a wrong inclusive slice here leaks test rows into training data — a classic silent model-quality bug.

## 📌 Summary

| Method / Attribute | What it does | Example |
|---|---|---|
| `pd.DataFrame(dict_of_lists)` | Build a table | `pd.DataFrame({"name": [...], "age": [...]})` |
| `head/tail/sample` | Peek at rows | `df.head(3)`, `df.sample(5, random_state=42)` |
| `shape`, `columns`, `dtypes`, `info()` | Structure report | `df.info()` |
| `df["col"]` / `df[["a","b"]]` | Column → Series / subset DataFrame | `df[["name", "score"]]` |
| `df["new"] = expr` | Add derived column | `df["total"] = df["price"] * df["qty"]` |
| `rename(columns={})` | Rename columns | `df.rename(columns={"old": "new"})` |
| `drop(col, axis=1)` / `drop(idx)` | Delete column / row | `df.drop("temp", axis=1)` |
| `loc[rows, cols]` | Label-based; slices inclusive; masks; assignment | `df.loc[df.score > 80, "name"]` |
| `iloc[i, j]` | Position-based; slices exclusive | `df.iloc[:3, 0]` |
| `set_index("col")` | Make a column the row labels | `df.set_index("student_id")` |

**Key takeaways**

- A DataFrame is a dict of Series sharing one index — mixed dtypes allowed per column, never within one.
- One bracket = Series, bracket-in-bracket = DataFrame; `drop`/`rename` return copies, so reassign.
- `.loc` = labels (inclusive slices), `.iloc` = positions (exclusive); conditional writes go through `.loc`.
- Copy-on-Write killed chained assignment — if a change "doesn't stick", you probably wrote through a chain.

## 🔗 Next Lesson

Next up: **[03_Read_Write_Files](../03_Read_Write_Files/notes.ipynb)** — stop hand-typing tables: load them from CSV and JSON files, and save your results back out.